# Nowcasting Model: Testing Same-Month-Last-Year (Lag12) Feature

Notebook 15a found that adding `delay_rate_lag12` (same month last year) to the **forecasting** model produced strong improvements (Ridge R2 +0.062, RF R2 +0.045). This notebook tests whether lag12 also improves the **nowcasting** model from notebook 8b, which already includes same-month weather features.


## Reference

| Source | Ridge R2 | RF R2 | XGB F1 |
|--------|----------|-------|--------|
| Nowcasting 8b (baseline) | 0.462 | 0.486 | 0.733 |
| Forecasting 15a + lag12 | 0.492 | 0.506 | 0.737 |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.linewidth'] = 1.5

try:
    import xgboost as xgb
    HAS_XGB = True
    print("XGBoost available")
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed")

%matplotlib inline

## 1. Data Preparation

Identical filtering to notebook 8b.

In [ ]:
df = pd.read_csv('../data/processed/ml_training_data_multiroute_hols.csv')

df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['month_num'] = df['year_month_dt'].dt.month
df['year'] = df['year'].astype(int)
df['airline_route'] = df['airline'] + '_' + df['departing_port'] + '_' + df['arriving_port']
df['route'] = df['departing_port'] + '_' + df['arriving_port']
df = df.sort_values(['airline_route', 'year_month_dt']).reset_index(drop=True)

print("Original shape: {}".format(df.shape))

In [ ]:
# Filter (same as 8b)
volume_threshold = 40
airline_route_volume = df.groupby('airline_route')['sectors_scheduled'].mean()
high_volume_ar = airline_route_volume[airline_route_volume >= volume_threshold].index.tolist()
df_filtered = df[df['airline_route'].isin(high_volume_ar)].copy()

anomalous_routes = ['Melbourne_Hobart', 'Adelaide_Sydney', 'Perth_Brisbane']
df_filtered = df_filtered[~df_filtered['route'].isin(anomalous_routes)].copy()
print("Records after filtering: {}".format(len(df_filtered)))

## 2. Feature Engineering

In [ ]:
# Standard lag features (same as 8b)
df_filtered['delay_rate_lag1'] = df_filtered.groupby('airline_route')['delay_rate'].shift(1)
df_filtered['delay_rate_lag2'] = df_filtered.groupby('airline_route')['delay_rate'].shift(2)
df_filtered['delay_rate_gradient'] = df_filtered['delay_rate_lag1'] - df_filtered['delay_rate_lag2']

# NEW: lag12
df_filtered['delay_rate_lag12'] = df_filtered.groupby('airline_route')['delay_rate'].shift(12)

# Cyclical month encoding
df_filtered['month_sin'] = np.sin(2 * np.pi * df_filtered['month_num'] / 12)
df_filtered['month_cos'] = np.cos(2 * np.pi * df_filtered['month_num'] / 12)

# One-hot encoding
airline_dummies = pd.get_dummies(df_filtered['airline'], prefix='airline')
df_filtered = pd.concat([df_filtered, airline_dummies], axis=1)
airline_cols = list(airline_dummies.columns)

route_dummies = pd.get_dummies(df_filtered['route'], prefix='route')
df_filtered = pd.concat([df_filtered, route_dummies], axis=1)
route_cols = list(route_dummies.columns)

# Weather features (same as 8b)
df_filtered['temp_volatility_total'] = df_filtered['temp_volatility_dep'] + df_filtered['temp_volatility_arr']
df_filtered['extreme_weather_days_total'] = df_filtered['extreme_weather_days_dep'] + df_filtered['extreme_weather_days_arr']
rainy_max = df_filtered['rainy_days_arr'].max()
temp_vol_max = df_filtered['temp_volatility_total'].max()
df_filtered['rainy_days_arr_exp'] = np.exp(df_filtered['rainy_days_arr'] / rainy_max)
df_filtered['temp_volatility_total_exp'] = np.exp(df_filtered['temp_volatility_total'] / temp_vol_max)

print("Airlines: {}".format(len(airline_cols)))
print("Routes:   {}".format(len(route_cols)))

## 3. Correlation Analysis

In [ ]:
corr_cols = ['delay_rate', 'delay_rate_lag1', 'delay_rate_lag12', 'delay_rate_gradient',
             'rainy_days_arr_exp', 'temp_volatility_total_exp', 'extreme_weather_days_total']
corrs = df_filtered[corr_cols].corr().loc['delay_rate'].drop('delay_rate')

print("Correlation with delay_rate:")
print("-" * 45)
for feat, val in corrs.sort_values(ascending=False).items():
    print("  {:<35} {:.4f}".format(feat, val))

In [ ]:
# Scatter: lag12 vs target, coloured by route
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

valid = df_filtered.dropna(subset=['delay_rate', 'delay_rate_lag1', 'delay_rate_lag12'])

ax = axes[0]
ax.scatter(valid['delay_rate_lag1'], valid['delay_rate'], alpha=0.3, s=10)
ax.plot([0, 0.6], [0, 0.6], 'r--')
ax.set_xlabel('delay_rate_lag1')
ax.set_ylabel('delay_rate')
ax.set_title('Lag1 vs Target (r = {:.3f})'.format(corrs['delay_rate_lag1']))
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 0.6); ax.set_ylim(0, 0.6)

ax = axes[1]
ax.scatter(valid['delay_rate_lag12'], valid['delay_rate'], alpha=0.3, s=10)
ax.plot([0, 0.6], [0, 0.6], 'r--')
ax.set_xlabel('delay_rate_lag12')
ax.set_ylabel('delay_rate')
ax.set_title('Lag12 vs Target (r = {:.3f})'.format(corrs['delay_rate_lag12']))
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 0.6); ax.set_ylim(0, 0.6)

plt.tight_layout()
plt.show()

## 4. Data Loss Comparison

In [ ]:
df_baseline = df_filtered.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient']).copy()
df_with_lag12 = df_filtered.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient', 'delay_rate_lag12']).copy()

print("Data availability:")
print("  Baseline (8b):   {:,} rows".format(len(df_baseline)))
print("  With lag12:      {:,} rows".format(len(df_with_lag12)))
print("  Rows lost:       {:,} ({:.1f}%)".format(
    len(df_baseline) - len(df_with_lag12),
    (len(df_baseline) - len(df_with_lag12)) / len(df_baseline) * 100
))

## 5. Train/Val/Test Split

Use the lag12-available subset for fair comparison between baseline and with-lag12.

In [ ]:
df_clean = df_with_lag12.copy()

train_mask = (((df_clean['year'] <= 2017)) | (df_clean['year'] == 2023))
val_mask = ((df_clean['year'] == 2018) | (df_clean['year'] == 2024))
test_mask = ((df_clean['year'] == 2019) | (df_clean['year'] >= 2025))

print("Train: {:,}".format(train_mask.sum()))
print("Val:   {:,}".format(val_mask.sum()))
print("Test:  {:,}".format(test_mask.sum()))

In [ ]:
# Feature sets
base_features = airline_cols + route_cols + ['month_sin', 'month_cos', 'delay_rate_lag1', 'sectors_scheduled']
weather_features = ['rainy_days_arr_exp', 'delay_rate_gradient', 'temp_volatility_total_exp', 'extreme_weather_days_total']
holiday_features = ['n_public_holidays_total', 'pct_school_holiday']

# Baseline: 8b feature set (38 features)
features_baseline = base_features + weather_features + holiday_features

# With lag12: 8b + lag12 (39 features)
features_with_lag12 = base_features + weather_features + ['delay_rate_lag12'] + holiday_features

print("Baseline features: {}".format(len(features_baseline)))
print("With lag12:        {}".format(len(features_with_lag12)))

## 6. Model Comparison

In [ ]:
# Tune Ridge alpha on validation set (RMSE)
alphas = np.logspace(-2, 4, 13)

def tune_ridge_alpha(df, features, train_mask, val_mask, alphas):
    X_train = df.loc[train_mask, features].values
    X_val = df.loc[val_mask, features].values
    y_train = df.loc[train_mask, 'delay_rate'].values
    y_val = df.loc[val_mask, 'delay_rate'].values

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    rows = []
    for a in alphas:
        ridge = Ridge(alpha=a)
        ridge.fit(X_train_scaled, y_train)
        pred = ridge.predict(X_val_scaled)
        rmse = np.sqrt(mean_squared_error(y_val, pred))
        r2 = r2_score(y_val, pred)
        rows.append({'alpha': a, 'rmse': rmse, 'r2': r2})

    results = pd.DataFrame(rows).sort_values('rmse')
    best_alpha = float(results.iloc[0]['alpha'])
    return best_alpha, results

ridge_alpha_baseline, ridge_tune_baseline = tune_ridge_alpha(
    df_clean, features_baseline, train_mask, val_mask, alphas
)
ridge_alpha_lag12, ridge_tune_lag12 = tune_ridge_alpha(
    df_clean, features_with_lag12, train_mask, val_mask, alphas
)

print("Best alpha (baseline):", ridge_alpha_baseline)
print("Best alpha (lag12):   ", ridge_alpha_lag12)

# Plot validation R2 vs alpha for lag12
ridge_tune_lag12_plot = ridge_tune_lag12.sort_values('alpha')
plt.figure(figsize=(6, 4))
plt.semilogx(ridge_tune_lag12_plot['alpha'], ridge_tune_lag12_plot['r2'], marker='o')
plt.axvline(ridge_alpha_lag12, color='red', linestyle='--', linewidth=1)
plt.xlabel('Alpha')
plt.ylabel('Validation R2')
plt.title('Ridge Alpha Tuning (Lag12)')
plt.grid(True, which='both', linestyle='--', alpha=0.4)
plt.show()



In [ ]:
def evaluate_model(df, features, train_mask, val_mask, test_mask, name, ridge_alpha=100):
    """Train and evaluate Ridge, RF, and XGBoost."""
    X_train = df.loc[train_mask, features].values
    X_val = df.loc[val_mask, features].values
    X_test = df.loc[test_mask, features].values

    y_train = df.loc[train_mask, 'delay_rate'].values
    y_val = df.loc[val_mask, 'delay_rate'].values
    y_test = df.loc[test_mask, 'delay_rate'].values

    y_train_clf = df.loc[train_mask, 'is_high_delay'].values
    y_val_clf = df.loc[val_mask, 'is_high_delay'].values
    y_test_clf = df.loc[test_mask, 'is_high_delay'].values

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    results = {'name': name}

    # Ridge
    ridge = Ridge(alpha=ridge_alpha)
    ridge.fit(X_train_scaled, y_train)
    ridge_pred = ridge.predict(X_test_scaled)
    results['ridge_r2'] = r2_score(y_test, ridge_pred)
    results['ridge_rmse'] = np.sqrt(mean_squared_error(y_test, ridge_pred))

    # Random Forest Regression
    rf = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    results['rf_r2'] = r2_score(y_test, rf_pred)
    results['rf_rmse'] = np.sqrt(mean_squared_error(y_test, rf_pred))
    results['rf_model'] = rf

    # XGBoost Classification
    if HAS_XGB:
        xgb_clf = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                                     min_child_weight=5, random_state=42, n_jobs=-1)
        xgb_clf.fit(X_train, y_train_clf, eval_set=[(X_val, y_val_clf)], verbose=False)
        xgb_pred = xgb_clf.predict(X_test)
        results['xgb_f1'] = f1_score(y_test_clf, xgb_pred)
    else:
        results['xgb_f1'] = np.nan

    return results, features


In [ ]:
print("Training models...")
print()

results_baseline, _ = evaluate_model(df_clean, features_baseline, train_mask, val_mask, test_mask, "Baseline (8b)", ridge_alpha=ridge_alpha_baseline)
results_lag12, _ = evaluate_model(df_clean, features_with_lag12, train_mask, val_mask, test_mask, "With Lag12", ridge_alpha=ridge_alpha_lag12)

print("=" * 80)
print("RESULTS COMPARISON")
print("=" * 80)
print()
header = "{:<15} {:>10} {:>12} {:>10} {:>12} {:>10}".format(
    'Model', 'Ridge R2', 'Ridge RMSE', 'RF R2', 'RF RMSE', 'XGB F1')
print(header)
print("-" * 75)

for r in [results_baseline, results_lag12]:
    xgb_str = "{:.4f}".format(r['xgb_f1']) if not np.isnan(r['xgb_f1']) else 'N/A'
    print("{:<15} {:>10.4f} {:>12.4f} {:>10.4f} {:>12.4f} {:>10}".format(
        r['name'], r['ridge_r2'], r['ridge_rmse'], r['rf_r2'], r['rf_rmse'], xgb_str))

print()
print("Delta (Lag12 - Baseline):")
print("  Ridge R2:   {:+.4f}".format(results_lag12['ridge_r2'] - results_baseline['ridge_r2']))
print("  Ridge RMSE: {:+.4f}".format(results_lag12['ridge_rmse'] - results_baseline['ridge_rmse']))
print("  RF R2:      {:+.4f}".format(results_lag12['rf_r2'] - results_baseline['rf_r2']))
print("  RF RMSE:    {:+.4f}".format(results_lag12['rf_rmse'] - results_baseline['rf_rmse']))
if HAS_XGB:
    print("  XGB F1:     {:+.4f}".format(results_lag12['xgb_f1'] - results_baseline['xgb_f1']))


## 7. Feature Importance

In [ ]:
rf_model = results_lag12['rf_model']
importance_df = pd.DataFrame({
    'Feature': features_with_lag12,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance (RF with Lag12 - Nowcasting):")
print("-" * 55)
for i, row in importance_df.head(15).iterrows():
    marker = " <-- NEW" if row['Feature'] == 'delay_rate_lag12' else ""
    print("  {:<35} {:.4f}{}".format(row['Feature'], row['Importance'], marker))

lag12_rank = list(importance_df['Feature']).index('delay_rate_lag12') + 1
lag12_importance = importance_df[importance_df['Feature'] == 'delay_rate_lag12']['Importance'].values[0]
print()
print("Lag12 rank: {} (importance: {:.4f})".format(lag12_rank, lag12_importance))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

top_features = importance_df.head(12)
y_pos = np.arange(len(top_features))
colors = ['#e74c3c' if f == 'delay_rate_lag12' else 'steelblue' for f in top_features['Feature']]

ax.barh(y_pos, top_features['Importance'], color=colors, alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(top_features['Feature'])
ax.invert_yaxis()
ax.set_xlabel('Importance')
ax.set_title('Feature Importances: Nowcasting with Lag12')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 8. Route-Level Performance

In [ ]:
# Predictions for route-level analysis
data_baseline = {}
data_lag12 = {}

# Baseline Ridge
scaler_b = StandardScaler()
X_train_b = scaler_b.fit_transform(df_clean.loc[train_mask, features_baseline].values)
X_test_b = scaler_b.transform(df_clean.loc[test_mask, features_baseline].values)
ridge_b = Ridge(alpha=100)
ridge_b.fit(X_train_b, df_clean.loc[train_mask, 'delay_rate'].values)

# Lag12 Ridge
scaler_l = StandardScaler()
X_train_l = scaler_l.fit_transform(df_clean.loc[train_mask, features_with_lag12].values)
X_test_l = scaler_l.transform(df_clean.loc[test_mask, features_with_lag12].values)
ridge_l = Ridge(alpha=100)
ridge_l.fit(X_train_l, df_clean.loc[train_mask, 'delay_rate'].values)

df_test = df_clean[test_mask].copy()
df_test['baseline_pred'] = ridge_b.predict(X_test_b)
df_test['lag12_pred'] = ridge_l.predict(X_test_l)

print("Route-Level Performance (Ridge R2):")
print("=" * 70)
header = "{:<20} {:>12} {:>12} {:>10}".format('Route', 'Baseline', 'With Lag12', 'Delta')
print(header)
print("-" * 60)

route_results = []
for route in sorted(df_test['route'].unique()):
    rd = df_test[df_test['route'] == route]
    r2_b = r2_score(rd['delay_rate'], rd['baseline_pred'])
    r2_l = r2_score(rd['delay_rate'], rd['lag12_pred'])
    delta = r2_l - r2_b
    print("{:<20} {:>12.4f} {:>12.4f} {:>+10.4f}".format(route, r2_b, r2_l, delta))
    route_results.append({'Route': route, 'Baseline': r2_b, 'With_Lag12': r2_l, 'Delta': delta})

route_df = pd.DataFrame(route_results)
print("-" * 60)
overall_b = r2_score(df_test['delay_rate'], df_test['baseline_pred'])
overall_l = r2_score(df_test['delay_rate'], df_test['lag12_pred'])
print("{:<20} {:>12.4f} {:>12.4f} {:>+10.4f}".format('OVERALL', overall_b, overall_l, overall_l - overall_b))

In [ ]:
# Visualize route-level deltas
fig, ax = plt.subplots(figsize=(14, 5))

routes = route_df['Route'].values
x = np.arange(len(routes))
deltas = route_df['Delta'].values
colors = ['#27ae60' if d > 0 else '#e74c3c' for d in deltas]

ax.bar(x, deltas, color=colors, alpha=0.8)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_xticks(x)
route_labels = [r.replace('_', ' to ') for r in routes]
ax.set_xticklabels(route_labels, rotation=45, ha='right')
ax.set_ylabel('Delta R2 (Lag12 - Baseline)')
ax.set_title('Route-Level Impact of Adding Lag12 to Nowcasting Model')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

improved = (route_df['Delta'] > 0).sum()
print("{} of {} routes improved with lag12".format(improved, len(route_df)))

## 9. Summary and Observations

In [ ]:
print("=" * 70)
print("LAG12 ON NOWCASTING MODEL - EVALUATION")
print("=" * 70)
print()

delta_ridge = results_lag12['ridge_r2'] - results_baseline['ridge_r2']
delta_rf = results_lag12['rf_r2'] - results_baseline['rf_r2']
delta_xgb = results_lag12['xgb_f1'] - results_baseline['xgb_f1'] if HAS_XGB else 0
delta_ridge_rmse = results_lag12['ridge_rmse'] - results_baseline['ridge_rmse']
delta_rf_rmse = results_lag12['rf_rmse'] - results_baseline['rf_rmse']

print("PERFORMANCE IMPACT (Nowcasting):")
print("  Ridge R2:   {:+.4f}  ({:.4f} -> {:.4f})".format(delta_ridge, results_baseline['ridge_r2'], results_lag12['ridge_r2']))
print("  Ridge RMSE: {:+.4f}  ({:.4f} -> {:.4f})".format(delta_ridge_rmse, results_baseline['ridge_rmse'], results_lag12['ridge_rmse']))
print("  RF R2:      {:+.4f}  ({:.4f} -> {:.4f})".format(delta_rf, results_baseline['rf_r2'], results_lag12['rf_r2']))
print("  RF RMSE:    {:+.4f}  ({:.4f} -> {:.4f})".format(delta_rf_rmse, results_baseline['rf_rmse'], results_lag12['rf_rmse']))
print("  XGB F1:     {:+.4f}  ({:.4f} -> {:.4f})".format(delta_xgb, results_baseline['xgb_f1'], results_lag12['xgb_f1']))
print()

print("COMPARISON WITH FORECASTING (notebook 15a):")
print("  Forecasting lag12 delta:  Ridge {:+.4f}, RF {:+.4f}, XGB {:+.4f}".format(+0.0619, +0.0446, +0.0056))
print("  Nowcasting lag12 delta:   Ridge {:+.4f}, RF {:+.4f}, XGB {:+.4f}".format(delta_ridge, delta_rf, delta_xgb))
print()

print("DATA COST:")
print("  Rows lost: {:,} ({:.1f}%)".format(
    len(df_baseline) - len(df_with_lag12),
    (len(df_baseline) - len(df_with_lag12)) / len(df_baseline) * 100
))
print()

print("FEATURE IMPORTANCE:")
print("  Lag12 rank: {} (importance: {:.4f})".format(lag12_rank, lag12_importance))
print()

### Observations
- Adding 12-month lagged delay rate as a feature provided a significant improvement (>5%) to the prediction performance: Ridge model R2 0.4664 -> 0.5219, RF model R2 .4852 -> 0.5270
    - `delay_rate_lag12` ranks second in feature importance, behind only `delay_rate_lag1`.
- Small improvement to the classification model (only XGBoost assessed in this notebook)


## 10. Next Step

The next notebook will explore the addition of passenger load factor as a feature. Such addition was explored in [notebook 12](/notebooks/12_feature_exploration_load_factor.ipynb), but has not been implemented to the latest models.